In [ ]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username= input("PostgreSQL user: "),
    password=getpass("Postgres password: "),   # typed at run time, never saved in the notebook
    host="localhost",
    port=5432,                                  # change if docker ps shows a different host port
    database="retail",
)
engine = create_engine(url)

print(pd.read_sql("SELECT COUNT(*) AS rows FROM transactions", engine))   # expect 1033068

      rows
0  1033068


In [2]:
purchases = pd.read_sql("""
    SELECT customer_id, invoice,
           MIN(invoice_date) AS invoice_date,
           SUM(revenue)      AS invoice_revenue
    FROM transactions
    WHERE has_customer AND NOT is_cancellation
    GROUP BY customer_id, invoice
    HAVING SUM(revenue) > 0
""", engine, parse_dates=["invoice_date"])

print(purchases.shape)
print("customers:", purchases["customer_id"].nunique())      # expect 5852
print(purchases["invoice_date"].min(), purchases["invoice_date"].max())

(36595, 4)
customers: 5852
2009-12-01 07:45:00 2011-12-09 12:50:00


In [3]:
def build_features(c, window=90):
    c = pd.Timestamp(c)
    past = purchases[purchases["invoice_date"] < c]
    fut = purchases[(purchases["invoice_date"] >= c) &
                    (purchases["invoice_date"] < c + pd.Timedelta(days=window))]

    g = past.groupby("customer_id")
    feat = pd.DataFrame({
        "recency_days": (c - g["invoice_date"].max()).dt.days,
        "tenure_days":  (c - g["invoice_date"].min()).dt.days,
        "frequency":    g.size(),
        "monetary":     g["invoice_revenue"].sum(),
    })
    feat["avg_order_value"] = feat["monetary"] / feat["frequency"]

    # recent activity
    for d in (90, 180):
        recent = past[past["invoice_date"] >= c - pd.Timedelta(days=d)]
        feat[f"purchases_last_{d}d"] = recent.groupby("customer_id").size()
        feat[f"revenue_last_{d}d"] = recent.groupby("customer_id")["invoice_revenue"].sum()
    recent_cols = [col for col in feat.columns if "last_" in col]
    feat[recent_cols] = feat[recent_cols].fillna(0)

    # average days between purchase days (NaN for customers with one purchase day)
    days = (past.assign(day=past["invoice_date"].dt.normalize())
            [["customer_id", "day"]].drop_duplicates()
            .sort_values(["customer_id", "day"]))
    gap = days.groupby("customer_id")["day"].diff().dt.days
    feat["avg_gap_days"] = gap.groupby(days["customer_id"]).mean()

    feat["label"] = feat.index.isin(fut["customer_id"].unique()).astype(int)
    feat["cutoff"] = c
    return feat.reset_index()


cutoff_list = ["2010-09-01", "2010-12-01", "2011-03-01", "2011-06-01", "2011-09-01"]
data = pd.concat([build_features(c) for c in cutoff_list], ignore_index=True)

print(data.shape)
print(data.groupby("cutoff")["label"].agg(rows="size", base_rate="mean").round(3))
print(data.isna().sum())

(22182, 13)
            rows  base_rate
cutoff                     
2010-09-01  3299      0.583
2010-12-01  4239      0.332
2011-03-01  4512      0.351
2011-06-01  4908      0.329
2011-09-01  5224      0.428
customer_id               0
recency_days              0
tenure_days               0
frequency                 0
monetary                  0
avg_order_value           0
purchases_last_90d        0
revenue_last_90d          0
purchases_last_180d       0
revenue_last_180d         0
avg_gap_days           7642
label                     0
cutoff                    0
dtype: int64


In [4]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

FEATURES = ["recency_days", "tenure_days", "frequency", "monetary", "avg_order_value",
            "purchases_last_90d", "revenue_last_90d",
            "purchases_last_180d", "revenue_last_180d", "avg_gap_days"]

train = data[data["cutoff"].isin(pd.to_datetime(["2010-09-01", "2010-12-01", "2011-03-01"]))]
val = data[data["cutoff"] == pd.Timestamp("2011-06-01")]
y_train, y_val = train["label"].values, val["label"].values
print(len(train), len(val), "val base rate:", round(y_val.mean(), 3))   # 12050, 4908, 0.329

def evaluate(name, y, score):
    base = y.mean()
    k = int(round(0.10 * len(y)))
    top = np.argsort(-score, kind="stable")[:k]
    prec_top = y[top].mean()
    return {"model": name,
            "PR-AUC": round(average_precision_score(y, score), 3),
            "ROC-AUC": round(roc_auc_score(y, score), 3),
            "top10%_precision": round(prec_top, 3),
            "top10%_lift": round(prec_top / base, 2),
            "max_possible_lift": round(1 / base, 2)}

12050 4908 val base rate: 0.329


In [5]:
results = []

# 1. recency alone: more recent = more likely to buy again
results.append(evaluate("Baseline: recency only", y_val, -val["recency_days"].values))

# 2. simple RFM-style rank score
rfm = ((-val["recency_days"]).rank(pct=True)
       + val["frequency"].rank(pct=True)
       + val["monetary"].rank(pct=True))
results.append(evaluate("Baseline: RFM rank score", y_val, rfm.values))

In [6]:
# Logistic regression: needs filled gaps and tamed skew
def prep_lr(df, gap_median):
    X = df[FEATURES].copy()
    X["gap_missing"] = X["avg_gap_days"].isna().astype(int)
    X["avg_gap_days"] = X["avg_gap_days"].fillna(gap_median)
    for c in ["frequency", "monetary", "avg_order_value", "revenue_last_90d", "revenue_last_180d"]:
        X[c] = np.log1p(X[c].clip(lower=0))
    return X

gap_median = train["avg_gap_days"].median()          # learned from train only
Xtr, Xva = prep_lr(train, gap_median), prep_lr(val, gap_median)
scaler = StandardScaler().fit(Xtr)
lr = LogisticRegression(max_iter=1000).fit(scaler.transform(Xtr), y_train)
results.append(evaluate("Logistic regression", y_val, lr.predict_proba(scaler.transform(Xva))[:, 1]))

# Gradient boosting: handles missing values and skew directly
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42)
gb.fit(train[FEATURES], y_train)
results.append(evaluate("Gradient boosting", y_val, gb.predict_proba(val[FEATURES])[:, 1]))

print(pd.DataFrame(results).to_string(index=False))

                   model  PR-AUC  ROC-AUC  top10%_precision  top10%_lift  max_possible_lift
  Baseline: recency only   0.587    0.774             0.662         2.01               3.04
Baseline: RFM rank score   0.706    0.809             0.872         2.65               3.04
     Logistic regression   0.728    0.820             0.882         2.68               3.04
       Gradient boosting   0.730    0.818             0.900         2.73               3.04


In [7]:
def rfm_score(df):
    return ((-df["recency_days"]).rank(pct=True)
            + df["frequency"].rank(pct=True)
            + df["monetary"].rank(pct=True)).values

def gb_scores(tr, te):
    m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                       max_iter=200, random_state=42)
    m.fit(tr[FEATURES], tr["label"])
    return m.predict_proba(te[FEATURES])[:, 1]

rows = []
for e in ["2010-12-01", "2011-03-01", "2011-06-01"]:
    e = pd.Timestamp(e)
    tr, te = data[data["cutoff"] < e], data[data["cutoff"] == e]
    y = te["label"].values
    for name, s in [("RFM rank score", rfm_score(te)), ("Gradient boosting", gb_scores(tr, te))]:
        r = evaluate(name, y, s)
        r.update(cutoff=e.date(), base_rate=round(y.mean(), 3))
        rows.append(r)

print(pd.DataFrame(rows)[["cutoff", "base_rate", "model", "PR-AUC", "ROC-AUC",
                          "top10%_lift", "max_possible_lift"]].to_string(index=False))

    cutoff  base_rate             model  PR-AUC  ROC-AUC  top10%_lift  max_possible_lift
2010-12-01      0.332    RFM rank score   0.676    0.785         2.52               3.01
2010-12-01      0.332 Gradient boosting   0.662    0.773         2.48               3.01
2011-03-01      0.351    RFM rank score   0.713    0.788         2.48               2.85
2011-03-01      0.351 Gradient boosting   0.705    0.772         2.54               2.85
2011-06-01      0.329    RFM rank score   0.706    0.809         2.65               3.04
2011-06-01      0.329 Gradient boosting   0.730    0.818         2.73               3.04


In [8]:
rng = np.random.default_rng(42)
s_rfm = rfm_score(val)
s_gb = gb.predict_proba(val[FEATURES])[:, 1]

diffs = []
for _ in range(500):
    i = rng.integers(0, len(y_val), len(y_val))
    diffs.append(average_precision_score(y_val[i], s_gb[i])
                 - average_precision_score(y_val[i], s_rfm[i]))

print("PR-AUC gain of gradient boosting over RFM (2.5%, median, 97.5%):",
      np.percentile(diffs, [2.5, 50, 97.5]).round(3))

PR-AUC gain of gradient boosting over RFM (2.5%, median, 97.5%): [0.016 0.024 0.033]


In [9]:
p_lr = lr.predict_proba(scaler.transform(Xva))[:, 1]
p_gb = gb.predict_proba(val[FEATURES])[:, 1]

for name, p in [("Logistic regression", p_lr), ("Gradient boosting", p_gb)]:
    d = pd.DataFrame({"p": p, "y": y_val})
    d["bin"] = pd.qcut(d["p"], 10, duplicates="drop")
    t = d.groupby("bin", observed=True).agg(avg_predicted=("p", "mean"),
                                            actual_rate=("y", "mean"),
                                            customers=("y", "size")).round(3)
    print(name)
    print(t.to_string())
    print("overall predicted:", round(p.mean(), 3), "| actual:", round(y_val.mean(), 3), "\n")

Logistic regression
                 avg_predicted  actual_rate  customers
bin                                                   
(0.0334, 0.108]          0.087        0.051        491
(0.108, 0.149]           0.129        0.088        491
(0.149, 0.186]           0.167        0.118        491
(0.186, 0.245]           0.213        0.139        490
(0.245, 0.298]           0.272        0.244        491
(0.298, 0.367]           0.329        0.285        491
(0.367, 0.478]           0.419        0.335        490
(0.478, 0.601]           0.538        0.507        491
(0.601, 0.743]           0.668        0.646        491
(0.743, 1.0]             0.851        0.882        491
overall predicted: 0.367 | actual: 0.329 

Gradient boosting
                                avg_predicted  actual_rate  customers
bin                                                                  
(0.036199999999999996, 0.0895]          0.061        0.049        491
(0.0895, 0.155]                         0.126    

In [10]:
data.to_csv("../data/processed/model_data.csv", index=False)